# Cálculo Analítico de Mareas: Puertos Secundarios

En los Anuarios de Mareas (como el del Instituto Hidrográfico de la Marina), sólo se publican predicciones completas para unos pocos **Puertos Patrón** (ej. Cádiz, Bilbao). Para un puerto cercano (Puerto Secundario), se calculan las diferencias horarias (adelanto o atraso) y las diferencias de alturas (ratio de amplitud).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import datetime

# Datos del Puerto Patrón (Extraídos del Anuario de Mareas, p.ej. Cádiz)
bajamar_patron_hora = "10:30"
bajamar_patron_altura = 0.8 # metros

pleamar_patron_hora = "16:45"
pleamar_patron_altura = 3.2 # metros

# Correcciones para el Puerto Secundario (Extraídas de las tablas de diferencias)
dif_hora_pleamar = -30 # minutos (Ocurre 30 mins antes)
dif_hora_bajamar = -25 # minutos

# En algunos libros se da Diferencia de Altura pura, en otros un multiplicador/ratio de amplitud.
# Supongamos diferencia aditiva directa:
dif_altura_pleamar = -0.4 # metros
dif_altura_bajamar = +0.1 # metros

# 1. Transformar tiempos a minutos desde medianoche
def time_to_mins(t_str):
    h, m = map(int, t_str.split(':'))
    return h * 60 + m

def mins_to_time(m_total):
    h = int(m_total // 60)
    m = int(m_total % 60)
    return f"{h:02d}:{m:02d}"

# Cálculos Puerto Secundario
bj_sec_mins = time_to_mins(bajamar_patron_hora) + dif_hora_bajamar
pl_sec_mins = time_to_mins(pleamar_patron_hora) + dif_hora_pleamar

bj_sec_alt = bajamar_patron_altura + dif_altura_bajamar
pl_sec_alt = pleamar_patron_altura + dif_altura_pleamar

print(f"--- PUERTO SECUNDARIO ---")
print(f"BAJAMAR: Hora {mins_to_time(bj_sec_mins)} | Altura: {bj_sec_alt:.2f} m")
print(f"PLEAMAR: Hora {mins_to_time(pl_sec_mins)} | Altura: {pl_sec_alt:.2f} m")

# Amplitud (A) y Duración (D)
amplitud = pl_sec_alt - bj_sec_alt
duracion = pl_sec_mins - bj_sec_mins
print(f"Amplitud de la marea: {amplitud:.2f} m")
print(f"Duración de la vaciante/creciente: {mins_to_time(duracion)} ({duracion} minutos)")


### Calcular sonda (profundidad) en un instante intermedio

Si queremos entrar a puerto a una hora concreta (ej. 14:00), usamos la fórmula matemática estándar aproximada (onda sinusoidal) conocida como regla de los doceavos, o directamente trigonométrica.

In [ ]:
# ¿Qué sonda tendremos a las 14:00?
hora_deseada = "14:00"
hd_mins = time_to_mins(hora_deseada)

# Fórmula Trigonométrica (Más precisa que los doceavos)
# Altura = Altura_Bajamar + Amplitud/2 * (1 - cos( pi * (T_deseada - T_bj) / Duracion ))

fase_rad = np.pi * (hd_mins - bj_sec_mins) / duracion
altura_momento = bj_sec_alt + (amplitud / 2.0) * (1 - np.cos(fase_rad))

print(f"\nAltura de marea calculada a las {hora_deseada}: {altura_momento:.2f} metros")

# Visualización Gráfica Marea
horas = np.linspace(bj_sec_mins, pl_sec_mins, 100)
alturas = bj_sec_alt + (amplitud / 2.0) * (1 - np.cos(np.pi * (horas - bj_sec_mins) / duracion))

plt.figure(figsize=(10,5))
plt.plot(horas / 60, alturas, label="Curva de Marea (Trigonométrica)", color='b')
plt.plot(hd_mins / 60, altura_momento, 'ro', markersize=10, label=f"Sonda a las {hora_deseada}")

plt.xlabel("Hora del Día (Decimal)")
plt.ylabel("Altura Marea (Metros)")
plt.title("Curva de Marea Puerto Secundario")
plt.grid(True)
plt.legend()
plt.show()
